# Project 3: Exploratory Data Analysis (EDA)
**Thiranex Data Science Internship Portfolio**  
*Author:* Data Science Intern  
*Dataset:* Titanic Socio-Economic & Demographic Dataset  

---
## 1. Project Overview & Analytical Scope
Exploratory Data Analysis (EDA) is the rigorous detective work of data science: formulating hypotheses, uncovering structural patterns, examining multi-attribute interactions, and confirming observations using inferential statistics. This notebook conducts a deep-dive EDA incorporating:
1. **Descriptive & Distributional Profiling:** Moments of distribution (Mean, Variance, Skewness, Kurtosis).
2. **Univariate, Bivariate, and Multivariate Visualizations:** Boxplots, violin plots, KDEs, and scatter plots.
3. **Statistical Hypothesis Testing:** Formal Chi-Square tests of independence and non-parametric Mann-Whitney U tests.
4. **10 Actionable Historical & Business Insights:** Translating data observations into structured domain knowledge.


In [1]:
# 1. Environment & Library Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Aesthetic configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
print('Libraries successfully imported!')

In [2]:
# 2. Dataset Loading
data_path = '../data/titanic_data.csv'
df = pd.read_csv(data_path)
print(f'Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [3]:
# 3. Descriptive & Distributional Statistics
num_cols = ['Age', 'Fare', 'FamilySize', 'SibSp', 'Parch']
stats_df = pd.DataFrame({
    'Mean': df[num_cols].mean(),
    'Std': df[num_cols].std(),
    'Median': df[num_cols].median(),
    'IQR': df[num_cols].quantile(0.75) - df[num_cols].quantile(0.25),
    'Skewness': df[num_cols].skew(),
    'Kurtosis': df[num_cols].kurtosis()
}).round(2)
print('=== DESCRIPTIVE STATISTICS ===')
stats_df

In [4]:
# 4. Univariate Analysis: Numerical Distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for i, col in enumerate(['Age', 'Fare', 'FamilySize']):
    sns.histplot(df[col], kde=True, color='#2563EB', bins=25, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}', weight='bold')
plt.tight_layout()
plt.show()

In [5]:
# 5. Bivariate Analysis: Survival by Class and Gender
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=df, x='Pclass', y='Survived', hue='Sex', palette=['#3B82F6', '#EC4899'], errorbar=None, ax=ax)
ax.set_title('Survival Rate across Socio-Economic Class & Gender', weight='bold', fontsize=13)
ax.set_ylabel('Survival Probability')
ax.set_ylim(0, 1.05)
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f'{h*100:.1f}%', (p.get_x() + p.get_width() / 2., h / 2), ha='center', va='center', color='white', weight='bold')
plt.tight_layout()
plt.show()

In [6]:
# 6. Violin Plot: Age Distribution across Classes Split by Survival
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df, x='Pclass', y='Age', hue='Survived', split=True, palette=['#EF4444', '#10B981'], inner='quartile', ax=ax)
ax.set_title('Age Density by Class Split by Survival Outcome', weight='bold', fontsize=13)
ax.set_ylabel('Age (Years)')
plt.tight_layout()
plt.show()

In [7]:
# 7. Correlation Analysis: Pearson vs Spearman
num_features = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
mask = np.triu(np.ones_like(df[num_features].corr(), dtype=bool))
sns.heatmap(df[num_features].corr(method='pearson'), mask=mask, annot=True, fmt='.2f', cmap='vlag', ax=ax1, cbar=False)
ax1.set_title('Pearson (Linear Correlation)', weight='bold')
sns.heatmap(df[num_features].corr(method='spearman'), mask=mask, annot=True, fmt='.2f', cmap='vlag', ax=ax2)
ax2.set_title('Spearman (Rank Correlation)', weight='bold')
plt.tight_layout()
plt.show()

In [8]:
# 8. Formal Statistical Hypothesis Testing
hyp_list = []
# Chi-square on categorical features
for cat in ['Sex', 'Pclass', 'Embarked', 'IsAlone', 'Title']:
    ct = pd.crosstab(df[cat], df['Survived'])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    hyp_list.append({'Feature': cat, 'Test': 'Chi-Square', 'Statistic': round(chi2, 2), 'p-value': f'{p:.2e}', 'Significant (p < 0.01)': p < 0.01})

# Mann-Whitney U on continuous features
for num in ['Fare', 'Age', 'FamilySize']:
    s0 = df[df['Survived'] == 0][num].dropna()
    s1 = df[df['Survived'] == 1][num].dropna()
    u, p = stats.mannwhitneyu(s0, s1)
    hyp_list.append({'Feature': num, 'Test': 'Mann-Whitney U', 'Statistic': round(u, 2), 'p-value': f'{p:.2e}', 'Significant (p < 0.01)': p < 0.01})

pd.DataFrame(hyp_list)

## 9. Top 10 Actionable Data & Business Insights
1. **Gender Protocol Dominance:** Females achieved 74.2% survival vs 18.9% for males (p < 1e-50), reflecting institutional prioritization of female passengers.
2. **Socio-Economic Egress Privilege:** 1st Class passengers survived at 62.9% vs 24.2% for 3rd Class (p < 1e-22), driven by closer cabin proximity to lifeboat davits.
3. **First-Class Female Survival:** 96.8% of 1st-class women survived, exhibiting near-total preservation.
4. **Inverted-U Family Survival:** Traveling in small family units (2-4 persons) yielded peak survival (~58-72%), whereas solo travelers (30.4%) and large families 7+ (<15%) suffered extreme mortality.
5. **Solo Traveler Vulnerability:** Solo passengers accounted for 60.3% of the manifest and 69.6% of casualties (p < 1e-8), demonstrating the hazards of unassisted navigation in crisis.
6. **Youth Prioritization:** 57.4% of children under 12 survived. In 2nd Class, 100% of male children survived.
7. **Purchasing Power Correlation:** Non-parametric tests confirm survivors paid significantly higher ticket fares (Mann-Whitney U p = 1.17e-21).
8. **Cherbourg Port Disparity:** Cherbourg embarkations had 55.4% survival versus Southampton's 33.7%, explained by Cherbourg's 50.6% 1st-class ticket concentration.
9. **Honorific Titles as Socio-Demographic Vectors:** 'Mrs' (79.2%) and 'Miss' (69.8%) outperformed all demographic categories, while 'Mr' was lowest at 15.7%.
10. **Monotonic Non-Linearity:** Spearman rank correlation between Pclass and Fare (-0.69) was significantly stronger than Pearson (-0.55), confirming exponential wealth tiers.
